# 3. Final labels and the submission

1. The **rules** (notebook 1) keep their H / A / D.
2. The **model** (notebook 2) fills the unlabelled frames with **H or A only**: it is good at
   the team (~96% right when it picks one) but calls too much live play dead, so D comes only
   from the rules.
3. **Viterbi** turns it into one clean label sequence: no flicker, a stoppage lasts at least 1 s.
4. Write the CSV, validate it, and check it once on the public 5 minutes.

The code lives in `possession/`.

In [14]:
import subprocess
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, 'evaluation')
import possession_metrics as pm

from possession import features, model, rules, submission, viterbi
from possession.data import (ATHLETIC, BARCA, DEAD, FPS, HALVES, LABEL_NAME, N_FRAMES, SECOND_HALF_KICKOFF,
                             UNLABELLED, graded_frames, load_events, load_tracking, runs_of)


def show(labels):
    return ''.join(LABEL_NAME[x] for x in labels)

## 1. Rule labels, features, model

Each in one call (notebooks 1 and 2 show the steps). The model is trained like notebook 2,
section 12: all labelled frames, except D frames only R2 / R5 mark and the 2nd half before kickoff.

In [15]:
tracking = load_tracking()
events = load_events()
graded = graded_frames()
labels, trusted_dead = rules.initial_labels(tracking, events)
X = features.player_features(tracking)

usable = (labels != UNLABELLED) & graded & ~((labels == DEAD) & ~trusted_dead)
usable[HALVES[1][1]:SECOND_HALF_KICKOFF] = False
proba = model.train(X, labels, usable).predict_proba(X)
unlabelled = graded & (labels == UNLABELLED)
print(f'unlabelled: {unlabelled[graded].mean():.1%} of graded frames')

unlabelled: 18.1% of graded frames


## 2. Model → H or A only

The model gives three probabilities per frame. Drop P(D) and rescale the other two:
`P(H only) = P(H) / (P(H) + P(A))`. Example: an unlabelled frame where the model's top answer is D.

In [16]:
p_barca = proba[:, BARCA] / (proba[:, BARCA] + proba[:, ATHLETIC])
example = np.flatnonzero(unlabelled & (proba.argmax(axis=1) == DEAD))[0]
print(f'frame {example}: P(H) {proba[example, BARCA]:.2f}, P(A) {proba[example, ATHLETIC]:.2f}, '
      f'P(D) {proba[example, DEAD]:.2f}  ->  P(H only) {p_barca[example]:.2f}, P(A only) {1 - p_barca[example]:.2f}')

frame 11185: P(H) 0.01, P(A) 0.09, P(D) 0.89  ->  P(H only) 0.13, P(A only) 0.87


## 3. Evidence per frame

What Viterbi gets for each frame, as probabilities for H, A, D:
- **rule label** → 0.95 for it, 0.025 for each of the others (strong, not absolute, so a
  one-frame flicker in the rules can still be smoothed);
- **unlabelled** → the model's H-only / A-only, and **0.001 for D** (D only from the rules).

Example: frames around the start of an unlabelled stretch.

In [17]:
ev = viterbi.evidence(labels, p_barca)
start = runs_of(unlabelled)[100][0]
rows = range(start - 3, start + 3)
pd.DataFrame({'rule label': [LABEL_NAME[labels[t]] for t in rows], 'H': ev[rows, BARCA],
              'A': ev[rows, ATHLETIC], 'D': ev[rows, DEAD]}, index=rows).round(3)

,rule label,H,A,D
120154,H,0.950,0.025,0.025
120155,H,0.950,0.025,0.025
120156,H,0.950,0.025,0.025
120157,-,0.999,0.000,0.001
120158,-,0.999,0.000,0.001
120159,-,0.999,0.000,0.001


## 4. Viterbi on a toy example

Viterbi picks the label sequence with the most evidence minus the costs of its changes:
H ↔ A costs `switch_cost`, entering or leaving D costs `dead_cost`, staying is free, and a
stoppage lasts at least `min_dead` frames. 20 frames, `.` = unlabelled with the model's P(H) below it:

In [18]:
toy_rules = 'HHHAHH..DDDD..AA.AAA'
toy_p_barca = np.array([0.5] * 6 + [0.7, 0.6] + [0.5] * 4 + [0.3, 0.2, 0.5, 0.5, 0.4, 0.5, 0.5, 0.5])
toy_labels = np.array([{'H': BARCA, 'A': ATHLETIC, 'D': DEAD}.get(c, UNLABELLED) for c in toy_rules])
toy_ev = viterbi.evidence(toy_labels, toy_p_barca)
print('rules    ', toy_rules)
print('best each', show(toy_ev.argmax(axis=1)))
print('viterbi  ', show(viterbi.viterbi(toy_ev, viterbi.change_costs(switch_cost=4, dead_cost=2, min_dead=3))))

rules     HHHAHH..DDDD..AA.AAA
best each HHHAHHHHDDDDAAAAAAAA
viterbi   HHHHHHHHDDDDAAAAAAAA


- **Frame 3**: one A frame inside H is not worth two switches → H.
- **Frames 6–7** lean H (0.7, 0.6) → H; **12–13** lean A and A follows → A, switched right after
  the stoppage, where the change is cheapest.
- **Frame 16** (P(H) 0.4) sits between A frames → A.

## 5. Choosing the switch cost

Too low and flicker comes back; too high and real short possessions vanish. Chosen without any
ground truth: run the whole match with several values and look at possessions per minute and how
many possessions are shorter than 2 s. `dead_cost` = 5 (D comes from strong rule evidence, so it
matters little); `min_dead` = 1 s.

In [19]:
DEAD_COST = 5
halves = [(first, last + 1) for _, first, last in HALVES]
minutes = graded.sum() / FPS / 60


def run(switch_cost):
    out = np.full(N_FRAMES, UNLABELLED, np.int8)
    for a, b in halves:
        out[a:b] = viterbi.viterbi(ev[a:b], viterbi.change_costs(switch_cost, DEAD_COST))
    return out


def stats(out):
    spells = [(x, y) for a, b in halves for t in (BARCA, ATHLETIC) for x, y in runs_of(out[a:b] == t)]
    seconds = np.array([y - x for x, y in spells]) / FPS
    ruled = (labels != UNLABELLED) & graded
    return {'possessions per min': len(spells) / minutes, 'median length (s)': np.median(seconds),
            'shorter than 2 s %': 100 * np.mean(seconds < 2), 'D %': 100 * np.mean(out[graded] == DEAD),
            'rule labels changed %': 100 * np.mean(out[ruled] != labels[ruled])}


pd.DataFrame({cost: stats(run(cost)) for cost in [2, 5, 10, 20, 40]}).T.rename_axis('switch cost').round(2)

,possessions per min,median length (s),shorter than 2 s %,D %,rule labels changed %
switch cost,,,,,
2,3.51,6.28,19.35,39.5,0.02
5,3.28,6.80,13.54,39.5,0.02
10,3.06,8.08,9.88,39.5,0.02
20,2.95,8.32,8.65,39.5,0.02
40,2.87,8.52,7.57,39.5,0.02


From 2 to 10 the flicker drops quickly; above 10 almost nothing changes. **10** is where it
settles: a switch then needs ~3 frames of clear rule evidence or ~0.5 s of model evidence
leaning 70/30.

In [20]:
SWITCH_COST = 10
final = run(SWITCH_COST)
pd.Series(stats(final)).round(2)

possessions per min       3.06
median length (s)         8.08
shorter than 2 s %        9.88
D %                      39.50
rule labels changed %     0.02
dtype: float64

## 6. What the result looks like

The first possessions and stoppages of the match, with their length, and how the labels changed
compared to the rules:

In [21]:
first = HALVES[0][1]
edges = np.flatnonzero(np.diff(final[first:first + 90 * FPS])) + 1
bounds = np.r_[0, edges, 90 * FPS]
print('first 90 s:', '  '.join(f'{LABEL_NAME[final[first + a]]} {(b - a) / FPS:.1f}s' for a, b in zip(bounds[:-1], bounds[1:])))
print()
from_rule = pd.crosstab(pd.Series(labels[graded]).map(LABEL_NAME).rename('rules'),
                        pd.Series(final[graded]).map(LABEL_NAME).rename('final'))
from_rule

first 90 s: A 11.9s  H 9.2s  A 5.2s  H 23.7s  A 1.8s  H 9.1s  A 6.0s  H 23.1s



final,A,D,H
rules,,,
-,15083,0,13621
A,25422,0,0
D,14,62741,11
H,0,0,41965


## 7. Write and validate the CSV

Put your team name in `NAME`: the scorer names the row after the file.

In [22]:
NAME = 'The press breakers'
path = f'evaluation/results/{NAME}.csv'
submission.write_csv(final, graded, path)
print(subprocess.run([sys.executable, 'evaluation/validate_submission.py', path],
                     capture_output=True, text=True).stdout)

OK    The press breakers.csv
      199319 rows, frame 0..199318

      label counts
        H    55597   27.9%
        A    40519   20.3%
        D    62741   31.5%
        X    40462   20.3%

      reports 324 possessions of mean 11.9 s
      (counted with each X as a gap; the scorer bridges abstentions inside a real
       possession, so its count can be lower than this one)

Valid. Note this checks the format only -- it says nothing about whether the labels are right.
Score it against the public ground truth with run_evaluation.py.



## 8. One check on the public 5 minutes

**Correctness check only**: the settings above were chosen before this cell was run, and must not
be changed because of it (RULES.md). 5 minutes ≈ 17 possessions, so the score is noisy. The
starter baselines are scored with the same code for comparison.

In [23]:
gt = pm.label_codes(pm.read_label_csv('evaluation/gt/BAR-ATH_possession_gt_public.csv'))
rows = []
for name in [NAME, 'baseline_smooth25', 'baseline_smooth50', 'baseline_smooth125']:
    r = pm.evaluate(gt, pm.label_codes(pm.read_label_csv(f'evaluation/results/{name}.csv')), bootstrap_n=0)
    rows.append({'submission': name, 'spellF1': r['spell_f1_mean'], 'macroF1': r['macro_f1'],
                 'possessions (true)': r['n_gt_spells'], 'possessions (predicted)': r['n_pred_spells']})
pd.DataFrame(rows).round(3)

,submission,spellF1,macroF1,possessions (true),possessions (predicted)
0,The press breakers,0.469,0.738,17,18
1,baseline_smooth25,0.287,0.620,17,36
2,baseline_smooth50,0.326,0.621,17,26
3,baseline_smooth125,0.424,0.581,17,16


## Summary

| | H | A | D | possessions / min | median possession |
|---|---|---|---|---|---|
| final (graded frames) | 35.0% | 25.5% | 39.5% | 3.06 | 8.1 s |

- **Rules decide D, the model only H or A.** The 18% unlabelled frames went 47% H / 53% A; no D
  was added. Viterbi changed 0.02% of the rule labels (25 D frames at stoppage edges).
- **Switch cost 10**, chosen from sanity statistics only (section 5): above 10 the output barely changes.
- **Public 5 minutes** (correctness check, not used to choose anything): spellF1 0.47, macroF1 0.74,
  18 possessions predicted vs 17 true. Best starter baseline: 0.42 / 0.58. On 17 possessions that
  gap is within the noise, so this says the pipeline works, not how much better it is.
- **Known weak spot:** live vs dead where the ball is missing rests on the rules (R2, R5) and cannot
  be measured on this match.
- **File:** `evaluation/results/<NAME>.csv`, valid, 199,319 rows; rerunning gives the identical file.